# **Preprocessing**

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
from sklearn.preprocessing import LabelEncoder
import re
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import chi2_contingency, spearmanr

In [21]:
df = pd.read_csv("merged_data.csv.gz")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 435 entries, TransactionID to DeviceInfo
dtypes: float64(400), int64(4), str(31)
memory usage: 1.9 GB


In [22]:
df = df.astype({col: 'float32' for col in df.select_dtypes(include='float64').columns})

Helper function - lấy từ file eda

In [23]:
def get_feature_lists(df: pd.DataFrame, target_col: str, id_col: str | None = None) -> tuple[list[str], list[str]]:
    excluded = {target_col}
    if id_col:
        excluded.add(id_col)

    numeric_cols = [col for col in df.select_dtypes(include=np.number).columns if col not in excluded]
    categorical_cols = [col for col in df.columns if col not in excluded and col not in numeric_cols]
    return numeric_cols, categorical_cols

## **1. Handle missing value**

- Các cột missing value ~ 99% và không có class signal, drop cột
- Các cột numerical có abs_missing_gap >= 0.02 và overall_missing_pct >= 0.05 -> tạo cột isna và fill giá trị -999 để mô hình học được đây là giá trị bất thường
- Cột categorical fill giá trị 'missing' vào NaN

In [24]:
cols_to_drop = ['id_24', 'id_25', 'id_07', 'id_08', 'id_21','id_26','id_22','id_23','id_27']

df.drop(columns=cols_to_drop, inplace=True)

In [25]:
def get_top_missing_features(
    df_train: pd.DataFrame,
    target_col: str = "isFraud",
    top_k: int = 60
) -> Tuple[List[str], Dict, pd.DataFrame]:

    if target_col not in df_train.columns:
        raise ValueError(f"Target column '{target_col}' not found!")

    fraud_mask = df_train[target_col] == 1
    rows = []

    for col in df_train.columns:
        if col == target_col:
            continue

        is_missing = df_train[col].isna()
        missing_pct = is_missing.mean()

        if missing_pct < 0.001:
            continue

        # Missing gap
        fraud_missing_pct = is_missing[fraud_mask].mean()
        nonfraud_missing_pct = is_missing[~fraud_mask].mean()
        abs_gap = abs(fraud_missing_pct - nonfraud_missing_pct)

        # Fraud rate gap
        fraud_rate_missing = df_train.loc[is_missing, target_col].mean() if is_missing.any() else 0.0
        fraud_rate_present = df_train.loc[~is_missing, target_col].mean()
        fraud_rate_gap = abs(fraud_rate_missing - fraud_rate_present)

  
        if pd.api.types.is_numeric_dtype(df_train[col]):
            temp = df_train[[col, target_col]].copy()
            temp[col] = temp[col].fillna(temp[col].median())
            corr = abs(spearmanr(temp[col], temp[target_col], nan_policy='omit')[0])
            if np.isnan(corr):
                corr = 0.0
        else:
            contingency = pd.crosstab(
                df_train[col].fillna("MISSING"), 
                df_train[target_col],
                dropna=False
            )
            if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                chi2, _, _, _ = chi2_contingency(contingency)
                n = df_train.shape[0]
                min_dim = min(contingency.shape) - 1
                corr = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0
            else:
                corr = 0.0

        score = (
            0.35 * abs_gap +
            0.30 * fraud_rate_gap +
            0.25 * corr +
            0.10 * missing_pct
        )

        rows.append({
            "feature": col,
            "score": score,
            "abs_gap": abs_gap,
            "fraud_rate_gap": fraud_rate_gap,
            "missing_pct": missing_pct,
            "signal_corr": corr,
            "dtype": str(df_train[col].dtype)
        })

    ranking_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    top_features = ranking_df.head(top_k)["feature"].tolist()

    impute_stats = {}
    for col in top_features:
        if pd.api.types.is_numeric_dtype(df_train[col]):
            impute_stats[col] = df_train[col].median()
        else:
            impute_stats[col] = "MISSING"

    return top_features, impute_stats, ranking_df


def preprocess_missing(
    df: pd.DataFrame,
    top_features: List[str],
    impute_stats: Dict,
    target_col: str = "isFraud"
) -> pd.DataFrame:

    df = df.copy()

    indicator_list = []
    for col in top_features:
        if col not in df.columns:
            continue
        indicator = df[col].isna().astype("int8")
        indicator.name = f"{col}_isna"
        indicator_list.append(indicator)

    if indicator_list:
        indicators_df = pd.concat(indicator_list, axis=1)
        df = pd.concat([df, indicators_df], axis=1)

    for col in top_features:
        if col not in df.columns:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            fill_val = impute_stats.get(col, -999)
            df[col] = df[col].fillna(fill_val)
        else:
            df[col] = df[col].astype("string").fillna("MISSING").astype("category")

    return df

In [26]:
top_features, impute_stats, ranking_df = get_top_missing_features(
    df, 
    target_col="isFraud", 
    top_k=80
)

df_clean = preprocess_missing(
    df, 
    top_features, 
    impute_stats
)

print(f"Số cột ban đầu: {df.shape[1]} -> Số cột sau xử lý missing: {df_clean.shape[1]}")

Số cột ban đầu: 426 -> Số cột sau xử lý missing: 506


## **Transformation**


In [27]:
class SkewedFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, log_cols: List[str] = None, clip_percentile: float = 0.99):
        self.log_cols = log_cols if log_cols else []
        self.clip_percentile = clip_percentile
        self.clip_thresholds_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        # Xác định ngưỡng cắt (clipping) cho các cột số
        numeric_cols = X.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            # Tính toán phân vị (ví dụ 99%) để làm ngưỡng cắt
            self.clip_thresholds_[col] = X[col].quantile(self.clip_percentile)
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        # 1. Clipping có kiểm soát: Giới hạn các giá trị cực đại
        for col, threshold in self.clip_thresholds_.items():
            if col in X.columns:
                X[col] = X[col].clip(upper=threshold)
        
        # 2. Log-like transform: Dùng log1p (log(1+x)) cho nhóm cực lệch
        # giúp xử lý tốt cả những điểm "spike-at-zero"
        for col in self.log_cols:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

In [28]:
class CategoricalLevelManager(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols: List[str], min_freq: float = 0.001):
        self.cat_cols = cat_cols
        self.min_freq = min_freq
        self.common_labels_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        for col in self.cat_cols:
            if col in X.columns:
                # Tính tần suất của từng nhãn
                freq = X[col].value_counts(normalize=True, dropna=True)
                # Chỉ giữ lại các nhãn có tần suất > min_freq
                common = freq[freq >= self.min_freq].index.tolist()
                self.common_labels_[col] = common
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                # Chuyển về kiểu chuỗi để xử lý nhất quán
                series = X[col].astype(str)
                
                # Xác định các vị trí thiếu dữ liệu (NaN ban đầu)
                is_null = X[col].isna() | (series == 'nan') | (series == 'None')
                
                # Xác định các nhãn phổ biến (đã học từ bước fit)
                common = self.common_labels_.get(col, [])
                is_common = series.isin(common)

                X[col] = np.where(is_null, "MISSING",
                                 np.where(is_common, series, "OTHER"))
        return X

In [29]:
numeric_cols, categorical_cols = get_feature_lists(df, target_col='isFraud')

# Khởi tạo các bước xử lý mới
skew_transformer = SkewedFeatureTransformer(
    log_cols=['TransactionAmt', 'C1', 'C2'], # Ví dụ các cột cực lệch từ EDA
    clip_percentile=0.99
)

cat_manager = CategoricalLevelManager(
    cat_cols=categorical_cols,
    min_freq=0.0005 # Tần suất tối thiểu để không bị coi là 'OTHER'
)

# Thực thi tuần tự
df_transformed = skew_transformer.fit_transform(df_clean)
df_transformed = cat_manager.fit_transform(df_transformed)

print(df_transformed.info())

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 506 entries, TransactionID to V215_isna
dtypes: float32(389), float64(6), int64(2), int8(80), str(29)
memory usage: 1.1 GB
None


## **Prunning**

In [30]:
def prune_highly_correlated_features(
    df: pd.DataFrame,
    target_col: str = "isFraud",
    corr_threshold: float = 0.85,
    v_only: bool = True
) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:

    df = df.copy()

    if v_only:
        numeric_cols = [col for col in df.columns if col.startswith("V") and col != target_col]
    else:
        numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
        if target_col in numeric_cols:
            numeric_cols.remove(target_col)
    
    if len(numeric_cols) < 2:
        return df, [], pd.DataFrame()

    corr_matrix = df[numeric_cols].corr(method="pearson").abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = upper.stack().reset_index()
    high_corr_pairs.columns = ["feature_1", "feature_2", "corr"]
    high_corr_pairs = high_corr_pairs[high_corr_pairs["corr"] >= corr_threshold]
    
    target_corr = {}
    for col in numeric_cols:
        target_corr[col] = abs(df[col].corr(df[target_col], method="spearman"))
    

    to_drop = set()
    processed = set()
    
    for _, row in high_corr_pairs.sort_values("corr", ascending=False).iterrows():
        f1, f2 = row["feature_1"], row["feature_2"]
        if f1 in to_drop or f2 in to_drop:
            continue
        
        # So sánh signal với target
        if target_corr.get(f1, 0) >= target_corr.get(f2, 0):
            to_drop.add(f2)
        else:
            to_drop.add(f1)
    
    df_pruned = df.drop(columns=list(to_drop))
    kept_features = [col for col in numeric_cols if col not in to_drop]
    
    print(f"Pruned {len(to_drop)} cột tương quan cao.")
    print(f"   - Giữ lại: {len(kept_features)} cột V*")
    print(f"   - Threshold: {corr_threshold}")

    summary = pd.DataFrame({
        "dropped_feature": list(to_drop),
        "reason": "high_correlation"
    })
    
    return df_pruned, kept_features, summary

In [31]:
df_pruned, kept_v_features, prune_summary = prune_highly_correlated_features(
    df_transformed,
    target_col="isFraud",
    corr_threshold=0.8,      
    v_only=True               
)

print(f"Final shape sau prune: {df_pruned.shape}")

e:\miniconda\envs\waitroom\Lib\site-packages\pandas\core\nanops.py:1661: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Pruned 281 cột tương quan cao.
   - Giữ lại: 118 cột V*
   - Threshold: 0.8
Final shape sau prune: (590540, 225)


## **Encoding**


In [32]:
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols: List[str]):
        self.cat_cols = cat_cols
        self.encoders_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        for col in self.cat_cols:
            if col in X.columns:
                le = LabelEncoder()
                # Ép kiểu string để tránh lỗi mix types
                le.fit(X[col].astype(str))
                self.encoders_[col] = le
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for col, le in self.encoders_.items():
            if col in X.columns:
                # Xử lý các giá trị mới xuất hiện ở test set (nếu có) bằng cách map về 'RARE'
                X[col] = X[col].astype(str).map(lambda s: s if s in le.classes_ else 'RARE')
                X[col] = le.transform(X[col])
        return X

In [33]:
# Giả sử bạn đã có df_eda_clean từ bước trước đó
# Bước này tiếp nối cell cuối cùng trong preprocessing.ipynb của bạn

# 1. Khởi tạo encoder với danh sách cột categorical đã lấy từ get_feature_lists
label_encoder = CategoricalEncoder(cat_cols=categorical_cols)

# 2. Fit trên dữ liệu đã được xử lý Missing/Rare
# Lưu ý: Phải fit trên dữ liệu đã qua bước Categorical_Missing_Processor 
# để encoder "học" được cả nhãn 'MISSING' và 'RARE'
label_encoder.fit(df_pruned)

# 3. Chuyển đổi dữ liệu sang dạng số
df_encoded = label_encoder.transform(df_pruned)

# Kiểm tra kết quả
print(f"Hình dạng dữ liệu sau encoding: {df_encoded.shape}")
display(df_encoded[categorical_cols].head())

Hình dạng dữ liệu sau encoding: (590540, 225)


,ProductCD,card4,card6,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,...,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,4,2,2,0,0,2,2,2,2,0,...,12,0,55,0,1,1,1,1,0,1
1,4,3,2,14,0,0,1,1,0,2,...,12,0,55,0,1,1,1,1,0,1
2,4,4,3,27,0,2,2,2,0,0,...,12,0,55,0,1,1,1,1,0,1
3,4,3,3,34,0,0,1,1,0,2,...,12,0,55,0,1,1,1,1,0,1
4,1,3,2,14,0,0,1,1,3,1,...,7,38,30,4,2,0,2,2,2,3
